In [1]:
import requests
import geopandas as gpd
from shapely.geometry import box

def recuperer_liens_dalles_ign(xmin, ymin, xmax, ymax):
    """
    Interroge le WFS de l'IGN pour récupérer les URL de téléchargement 
    des dalles LiDAR HD intersectant une bounding box (Lambert 93).
    """
    wfs_url = "https://data.geopf.fr/wfs/ows"
    
    # Création d'un filtre spatial BBOX en EPSG:2154
    bbox_str = f"{xmin},{ymin},{xmax},{ymax},EPSG:2154"
    
    params = {
        'service': 'WFS',
        'version': '2.0.0',
        'request': 'GetFeature',
        'typename': 'IGNF_LIDAR-HD_METADONNEE:metadata',
        'srsName': 'EPSG:2154',
        'bbox': bbox_str,
        'outputFormat': 'application/json'
    }
    
    print("Interrogation du catalogue IGN LiDAR HD...")
    response = requests.get(wfs_url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        features = data.get('features', [])
        print(f"{len(features)} dalle(s) trouvée(s) sur cette emprise.")
        
        dalles_info = []
        for feat in features:
            props = feat['properties']
            dalles_info.append({
                'nom_dalle': props.get('identifiant'),
                'url_laz': props.get('url_npl'),      # Lien vers le nuage .copc.laz
                'url_mnt': props.get('url_mnt')       # Lien vers le MNT .tif associé
            })
        return dalles_info
    else:
        print(f"Erreur lors de la requête WFS : {response.status_code}")
        return []

# --- Exemple d'utilisation ---
emprise_wim = (769000.0, 6895000.0, 771000.0, 6897000.0)
liens = recuperer_liens_dalles_ign(*emprise_wim)
for d in liens:
    print(f"Dalle : {d['nom_dalle']} -> LAZ : {d['url_laz']}")

Interrogation du catalogue IGN LiDAR HD...
16 dalle(s) trouvée(s) sur cette emprise.
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__LAZ_LAMB93_MD_2025-08-29/LHD_FXX_0768_6895_PTS_LAMB93_IGN69.copc.laz
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__LAZ_LAMB93_MD_2025-08-29/LHD_FXX_0768_6896_PTS_LAMB93_IGN69.copc.laz
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__LAZ_LAMB93_MD_2025-08-29/LHD_FXX_0768_6897_PTS_LAMB93_IGN69.copc.laz
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__LAZ_LAMB93_MD_2025-08-29/LHD_FXX_0768_6898_PTS_LAMB93_IGN69.copc.laz
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__LAZ_LAMB93_MD_2025-08-29/LHD_FXX_0769_6895_PTS_LAMB93_IGN69.copc.laz
Dalle : None -> LAZ : https://data.geopf.fr/telechargement/download/LiDARHD-NUALID/NUALHD_1-0__

In [ ]:
import os
import requests
from pathlib import Path

def recuperer_liens_dalles_ign(xmin, ymin, xmax, ymax):
    """
    Interroge le WFS de l'IGN pour récupérer les URL de téléchargement 
    des dalles LiDAR HD intersectant une bounding box (Lambert 93).
    """
    wfs_url = "https://data.geopf.fr/wfs/ows"
    bbox_str = f"{xmin},{ymin},{xmax},{ymax},EPSG:2154"
    
    params = {
        'service': 'WFS',
        'version': '2.0.0',
        'request': 'GetFeature',
        'typename': 'IGNF_LIDAR-HD_METADONNEE:metadata',
        'srsName': 'EPSG:2154',
        'bbox': bbox_str,
        'outputFormat': 'application/json'
    }
    
    print("🔍 Interrogation du catalogue IGN LiDAR HD (WFS)...")
    response = requests.get(wfs_url, params=params, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        features = data.get('features', [])
        print(f"✨ {len(features)} dalle(s) trouvée(s) sur cette emprise.")
        
        dalles_info = []
        for feat in features:
            props = feat['properties']
            dalles_info.append({
                'nom_dalle': props.get('identifiant'),
                'url_laz': props.get('url_npl'),      # Lien vers le nuage .copc.laz
                'url_mnt': props.get('url_mnt')       # Lien vers le MNT .tif associé
            })
        return dalles_info
    else:
        print(f"❌ Erreur lors de la requête WFS : {response.status_code}")
        return []

def telecharger_dalles_lidar(dalles_info, dossier_sortie="data/brut_laz/", type_flux='url_laz'):
    """
    Télécharge les fichiers (LAZ ou MNT TIF) à partir de la liste des dictionnaires 
    obtenue par la requête WFS en gérant les gros volumes (streaming).
    """
    # Création du dossier de destination s'il n'existe pas
    Path(dossier_sortie).mkdir(parents=True, exist_ok=True)
    fichiers_locaux = []
    
    for dalle in dalles_info:
        nom_dalle = dalle.get('nom_dalle', 'inconnu')
        url = dalle.get(type_flux)
        
        if not url:
            print(f"⚠️ Pas d'URL disponible pour la dalle {nom_dalle} ({type_flux})")
            continue
            
        # Extraction du nom du fichier à partir de l'URL
        nom_fichier = url.split('/')[-1]
        chemin_local = os.path.join(dossier_sortie, nom_fichier)
        
        # Vérification si le fichier existe déjà pour éviter de le retélécharger inutilement
        if os.path.exists(chemin_local):
            print(f"ℹ️ Fichier déjà présent localement : {nom_fichier} (Téléchargement ignoré)")
            fichiers_locaux.append(chemin_local)
            continue
            
        print(f"⬇️ Téléchargement de {nom_fichier}...")
        try:
            # Téléchargement par flux pour les gros fichiers binaires
            response = requests.get(url, stream=True, timeout=120)
            if response.status_code == 200:
                with open(chemin_local, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                print(f"✅ Enregistré avec succès : {chemin_local}")
                fichiers_locaux.append(chemin_local)
            else:
                print(f"❌ Erreur HTTP {response.status_code} lors du téléchargement de {url}")
        except Exception as e:
            print(f"❌ Erreur technique pour {nom_fichier} : {e}")
            
    return fichiers_locaux

# --- Exemple d'utilisation globale ---
# 1. Définition d'une emprise d'étude en Lambert 93 (ex: autour d'un échangeur)
xmin, ymin, xmax, ymax = (769000.0, 6895000.0, 770000.0, 6896000.0)
# 
# 2. Récupération des liens via le WFS IGN
liste_dalles = recuperer_liens_dalles_ign(xmin, ymin, xmax, ymax)
# 
# 3. Téléchargement effectif des fichiers .copc.laz (Nuages de points)
fichiers_laz = telecharger_dalles_lidar(liste_dalles, dossier_sortie="datas/brut_laz/", type_flux='url_laz')
# 
# 4. (Optionnel) Téléchargement direct des MNT .tif associés
fichiers_mnt = telecharger_dalles_lidar(liste_dalles, dossier_sortie="outs/mnt/", type_flux='url_mnt')

🔍 Interrogation du catalogue IGN LiDAR HD (WFS)...
✨ 9 dalle(s) trouvée(s) sur cette emprise.
⬇️ Téléchargement de LHD_FXX_0768_6895_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0768_6895_PTS_LAMB93_IGN69.copc.laz
⬇️ Téléchargement de LHD_FXX_0768_6896_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0768_6896_PTS_LAMB93_IGN69.copc.laz
⬇️ Téléchargement de LHD_FXX_0768_6897_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0768_6897_PTS_LAMB93_IGN69.copc.laz
⬇️ Téléchargement de LHD_FXX_0769_6895_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0769_6895_PTS_LAMB93_IGN69.copc.laz
⬇️ Téléchargement de LHD_FXX_0769_6896_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0769_6896_PTS_LAMB93_IGN69.copc.laz
⬇️ Téléchargement de LHD_FXX_0769_6897_PTS_LAMB93_IGN69.copc.laz...
✅ Enregistré avec succès : data/brut_laz/LHD_FXX_0769_6897_PTS_LAMB93_IGN